# VideoObjectSwap_WanGP_Colab — chạy WanGP (Wan2GP) trên Colab T4 để đổi sản phẩm trong video

## Vì sao đổi từ notebook tự viết (`VideoObjectSwap_WanVACE_Colab.ipynb`) sang cái này

Notebook trước tự viết node graph ComfyUI bằng tay để dùng Wan2.1 VACE, và qua 4 vòng sửa
(v1→v4) vẫn còn lỗi: hiểu sai ngữ nghĩa `control_masks`/`reference_image` của VACE, dán
sản phẩm mới sai cách, môi trường AnyDoor gãy 3 lần. Nguyên nhân gốc: **tự suy luận cơ chế
VACE từ đọc source code** thay vì dùng một công cụ đã làm đúng việc này.

[**WanGP** (deepbeepmeep/Wan2GP)](https://github.com/deepbeepmeep/Wan2GP) là app nguồn mở
(giao diện Gradio) chạy Wan2.1/2.2 VACE (và nhiều model khác) với công cụ **Mask
Generator / Matanyone** làm sẵn đúng việc: tải video mẫu lên, **click chọn vùng sản phẩm
cần đổi qua các khung hình**, xuất thẳng ra đúng 3 input mà VACE cần (control video, mask,
reference image) — không cần tự đoán toạ độ bbox hay tự viết code dán ảnh nữa.

Notebook này **port lại phần cài đặt Colab** từ
[Square-Zero-Labs/Wan2GP-on-Colab](https://github.com/Square-Zero-Labs/Wan2GP-on-Colab)
(Apache-2.0, đã ghi nhận **tự kiểm thử trên Colab T4 free tier**) — thay vì tự đoán cách
cài lại từ đầu (bài học từ AnyDoor: tự đoán môi trường tốn 3 vòng sửa lỗi). Đã đọc lại
từng bước trước khi đưa vào đây, không copy mù.

## Cách dùng sau khi chạy hết các cell cài đặt

Khi cell cuối in ra dòng `Running on public URL: https://xxxx.gradio.live`, bấm vào link
đó để mở giao diện WanGP. Theo đúng công thức "Replace an object while keeping the
background" (docs/VACE.md của WanGP), áp dụng cho bài toán đổi sản phẩm:

1. **Chọn model**: `Vace 1.3B` (khuyến nghị cho T4 15 GB — xem cảnh báo VRAM bên dưới).
   Có thể thử `Vace 14B` (bản GGUF lượng tử hoá) nếu muốn chất lượng cao hơn, nhưng dễ hết
   VRAM/RAM hơn trên T4.
2. **Control Video Process** = `Inpainting`, **Area Processed** = `Masked area`.
3. Mở **Mask Generator / Matanyone**: tải `video mẫu` lên (video có sản phẩm CŨ), click
   vào sản phẩm ở khung hình đầu, **Set Mask**, **Generate Video Matting**, rồi **Export
   to Control Video Input and Video Mask Input**.
4. **Reference Image** → chọn **Inject Landscapes / People / Objects**, tải ảnh sản phẩm
   MỚI lên.
5. **Prompt**: mô tả rõ cái gì GIỮ NGUYÊN và cái gì THAY, ví dụ:
   > Keep the building, camera angle, water effect and lighting exactly the same. Replace
   > the product with a pink oversized button-up shirt, photorealistic, high quality.
6. Bấm **Generate**.

Nếu sản phẩm mới bị lệch hình dạng/kích thước so với mask: dùng **Expand / Shrink Mask**
trong Mask Generator (tương đương `mo_rong_mask` ở notebook cũ, nhưng chỉnh bằng UI).

## ⚠️ CHƯA CHẠY THẬT trên T4 (bởi tôi)

Cấu hình cài đặt (uv thay vì pip, torch 2.10.0+cu128 giữ nguyên nếu Colab đã có bản dùng
được, vá `TkAgg`→`Agg` trong Matanyone, ghi đè `onnxruntime-gpu` sang bản ổn định thay vì
bản nightly trong `requirements.txt` gốc) đều lấy từ notebook cộng đồng đã ghi nhận **test
trên Colab T4 free tier** (changelog 2026-08-07) — mức tin cậy cao hơn các phần tôi tự viết
trước đây, nhưng vẫn khuyến nghị đọc log kỹ khi chạy lần đầu.

`requirements.txt` của WanGP rất nặng (nó là "super app" đa năng: video + ảnh + âm
thanh/TTS), nên **cài đặt có thể mất khá lâu** (không đo được con số chính xác) và tải
nhiều GB — dùng `uv` thay vì `pip` để đỡ chậm ở bước giải quyết phụ thuộc.

## Model & giấy phép

| Thành phần | Nguồn | License |
|---|---|---|
| WanGP (code) | [deepbeepmeep/Wan2GP](https://github.com/deepbeepmeep/Wan2GP) | xem LICENSE của repo |
| Cấu trúc cài đặt Colab | [Square-Zero-Labs/Wan2GP-on-Colab](https://github.com/Square-Zero-Labs/Wan2GP-on-Colab) | Apache-2.0 |
| Wan2.1/2.2 VACE (model) | [ali-vilab/VACE](https://github.com/ali-vilab/VACE) | Apache-2.0 |

WanGP còn cho chọn nhiều model khác (Flux, Qwen Image, HunyuanVideo...) trong UI — **mỗi
model có giấy phép riêng**, tự kiểm tra trong UI trước khi dùng thương mại.

## Yêu cầu GPU

Colab T4 (15 GB VRAM, ~12.7 GB RAM hệ thống, ~112 GB đĩa). Ghi chú từ notebook cộng đồng:
*"free Colab GPU runtimes usually provide only 15 GB of T4 VRAM, which is too small for
most Wan2GP checkpoints"* — dùng `Vace 1.3B` hoặc bản GGUF lượng tử hoá của `Vace 14B`,
độ phân giải 480p, và cờ `--profile 5` (tối thiểu hoá RAM, không phải VRAM — T4 trên Colab
hay hết RAM hệ thống trước khi hết VRAM).


## Bước 1 — Kiểm tra GPU

In [ ]:
# Buoc 1: xac nhan co GPU. Neu loi: Runtime > Change runtime type > chon GPU (T4) > Save,
# roi chay lai tu dau.
import subprocess

try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError(
        'Khong thay GPU. Vao Runtime > Change runtime type, chon GPU (T4), Save, roi chay lai.'
    ) from exc


## Bước 2 — Đường dẫn làm việc + lưu trữ (tuỳ chọn Google Drive)

In [ ]:
# Buoc 2: duong dan lam viec + luu tru lau dai (tuy chon qua Google Drive).
#
# WanGP tai model/checkpoint kha nang (10-20+ GB tuy model chon trong UI). Mac dinh luu
# trong o dia Colab (mat khi Runtime bi ngat/het gio) — dat USE_GOOGLE_DRIVE_DATA=True de
# luu vao Drive, khoi phai tai lai moi lan mo Colab.
from pathlib import Path

USE_GOOGLE_DRIVE_DATA = False    #@param {type:"boolean"}

DRIVE_MOUNT_POINT = Path('/content/drive')
WAN2GP_ROOT = Path('/content/Wan2GP').resolve()
DU_LIEU_TAM = Path('/content/Wan2GP-data').resolve()
DU_LIEU_DRIVE = (DRIVE_MOUNT_POINT / 'MyDrive' / 'Wan2GP-data').resolve()

if USE_GOOGLE_DRIVE_DATA:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT_POINT), force_remount=False)
    WAN_DATA_ROOT = DU_LIEU_DRIVE
    che_do = 'Google Drive (luu lau dai)'
else:
    WAN_DATA_ROOT = DU_LIEU_TAM
    che_do = 'o dia Colab (mat khi het phien)'

WAN_CKPTS_DIR = (WAN_DATA_ROOT / 'ckpts').resolve()
WAN_LORAS_DIR = (WAN_DATA_ROOT / 'loras').resolve()
WAN_OUTPUTS_DIR = (WAN_DATA_ROOT / 'outputs').resolve()
WAN_CACHE_DIR = (WAN_DATA_ROOT / 'cache').resolve()
for d in (WAN_CKPTS_DIR, WAN_LORAS_DIR, WAN_OUTPUTS_DIR, WAN_CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f'Luu tru: {che_do}')
print(f'  ckpts   -> {WAN_CKPTS_DIR}')
print(f'  loras   -> {WAN_LORAS_DIR}')
print(f'  outputs -> {WAN_OUTPUTS_DIR}')
print(f'  cache   -> {WAN_CACHE_DIR}')


## Bước 3 — Tải/cập nhật WanGP

In [ ]:
# Buoc 3: tai (hoac cap nhat) WanGP, roi noi thu muc ckpts/loras/outputs cua no vao
# WAN_DATA_ROOT (o dia Colab hoac Drive tuy buoc 2) bang symlink — de model tai ve khong
# mat khi xoa/clone lai repo.
import shutil, subprocess
from pathlib import Path


def gop_thu_muc(nguon: Path, dich: Path) -> None:
    for con in list(nguon.iterdir()):
        d = dich / con.name
        if d.exists():
            if con.is_dir() and d.is_dir():
                gop_thu_muc(con, d)
                con.rmdir()
                continue
            raise RuntimeError(f'Khong the chuyen {con} vao {dich}: {d} da ton tai.')
        shutil.move(str(con), str(d))


def gan_thu_muc_du_lieu(duong_dan_repo: Path, duong_dan_du_lieu: Path) -> None:
    duong_dan_du_lieu.mkdir(parents=True, exist_ok=True)
    if duong_dan_repo.is_symlink():
        if duong_dan_repo.resolve() != duong_dan_du_lieu.resolve():
            raise RuntimeError(f'{duong_dan_repo} dang tro toi {duong_dan_repo.resolve()}, '
                              f'khac voi {duong_dan_du_lieu} mong doi.')
        print(f'Da co lien ket: {duong_dan_repo} -> {duong_dan_du_lieu}')
        return
    if duong_dan_repo.exists():
        if not duong_dan_repo.is_dir():
            raise RuntimeError(f'Can mot thu muc tai {duong_dan_repo}.')
        gop_thu_muc(duong_dan_repo, duong_dan_du_lieu)
        duong_dan_repo.rmdir()
    else:
        duong_dan_repo.parent.mkdir(parents=True, exist_ok=True)
    duong_dan_repo.symlink_to(duong_dan_du_lieu, target_is_directory=True)
    print(f'Da lien ket {duong_dan_repo.name} -> {duong_dan_du_lieu}')


THU_MUC_QUAN_LY = {'ckpts', 'loras', 'outputs', 'ffmpeg_bins'}
REPO_URL = 'https://github.com/deepbeepmeep/Wan2GP.git'

if WAN2GP_ROOT.exists():
    chua_theo_doi = subprocess.run(
        ['git', '-C', str(WAN2GP_ROOT), 'ls-files', '--others', '--exclude-standard'],
        check=True, capture_output=True, text=True)
    file_cua_nguoi_dung = [
        l for l in chua_theo_doi.stdout.splitlines()
        if l.split('/', 1)[0] not in THU_MUC_QUAN_LY]
    if file_cua_nguoi_dung:
        xem_truoc = ', '.join(file_cua_nguoi_dung[:5]) + (' …' if len(file_cua_nguoi_dung) > 5 else '')
        print(f'Repo co file rieng cua ban ({xem_truoc}) -> bo qua cap nhat de giu lai.')
    else:
        print('Repo da co san -> cap nhat len ban moi nhat...')
        subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'checkout', '--', '.'], check=True)
        subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(WAN2GP_ROOT)], check=True)

gan_thu_muc_du_lieu(WAN2GP_ROOT / 'ckpts', WAN_CKPTS_DIR)
gan_thu_muc_du_lieu(WAN2GP_ROOT / 'loras', WAN_LORAS_DIR)
gan_thu_muc_du_lieu(WAN2GP_ROOT / 'outputs', WAN_OUTPUTS_DIR)


## Bước 4 — Cài thư viện hệ thống + ffmpeg

In [ ]:
# Buoc 4: cai thu vien he thong (video/audio) + kiem tra ffmpeg co ho tro co `-fps_mode`
# hay khong (ffmpeg co san tren mot so anh Colab qua cu, thieu co nay se lam WanGP loi
# ngam khi ghep video) — thieu thi tai ban tinh (static build) moi hon, co kiem tra SHA-256.
import hashlib, json, os, platform, re, shutil, subprocess, tarfile, tempfile, urllib.request
from pathlib import Path

GOI_APT = ['libglib2.0-0', 'libgl1', 'libportaudio2']
FFMPEG_RELEASE_API = 'https://api.github.com/repos/BtbN/FFmpeg-Builds/releases/latest'
FFMPEG_DOWNLOAD_PREFIX = 'https://github.com/BtbN/FFmpeg-Builds/releases/download/'
FFMPEG_BIN_DIR = WAN2GP_ROOT / 'ffmpeg_bins'
FFMPEG_CACHE_DIR = WAN_CACHE_DIR / 'downloads'
KIEN_TRUC_FFMPEG = {'x86_64': 'linux64', 'amd64': 'linux64', 'aarch64': 'linuxarm64', 'arm64': 'linuxarm64'}


def da_cai_dpkg(goi: str) -> bool:
    return subprocess.run(['dpkg', '-s', goi], capture_output=True).returncode == 0


thieu = [g for g in GOI_APT if not da_cai_dpkg(g)]
if thieu:
    env = os.environ.copy()
    env['DEBIAN_FRONTEND'] = 'noninteractive'
    print('Cai dat:', ', '.join(thieu))
    subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True, env=env)
    subprocess.run(['sudo', 'apt-get', 'install', '-y', '--no-install-recommends', *thieu], check=True, env=env)
else:
    print('Cac thu vien he thong da co san.')


def ffmpeg_du_tinh_nang(binary: Path) -> bool:
    try:
        r = subprocess.run([str(binary), '-hide_banner', '-h', 'full'], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, timeout=30)
    except (OSError, subprocess.TimeoutExpired):
        return False
    return r.returncode == 0 and re.search(r'^\s*-fps_mode(?:\[:[^]]+\])?(?:\s|$)', r.stdout, re.MULTILINE) is not None


def thuc_thi_duoc(binary: Path) -> bool:
    try:
        return subprocess.run([str(binary), '-version'], stdout=subprocess.DEVNULL,
                              stderr=subprocess.DEVNULL, timeout=30).returncode == 0
    except (OSError, subprocess.TimeoutExpired):
        return False


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def goi_github_json(url: str) -> dict:
    req = urllib.request.Request(url, headers={'Accept': 'application/vnd.github+json',
                                                'User-Agent': 'wangp-colab-vn', 'X-GitHub-Api-Version': '2022-11-28'})
    with urllib.request.urlopen(req, timeout=30) as resp:
        return json.load(resp)


def tai_file(url: str, dich: Path, kich_thuoc_du_kien: int) -> None:
    req = urllib.request.Request(url, headers={'User-Agent': 'wangp-colab-vn'})
    with urllib.request.urlopen(req, timeout=120) as resp, dich.open('wb') as out:
        da_tai, moc_bao = 0, 10
        while True:
            chunk = resp.read(1 << 20)
            if not chunk:
                break
            out.write(chunk)
            da_tai += len(chunk)
            if kich_thuoc_du_kien:
                pct = da_tai * 100 // kich_thuoc_du_kien
                if pct >= moc_bao:
                    print(f'Tai ffmpeg: {min(pct, 100)}%')
                    moc_bao += 10
    if kich_thuoc_du_kien and da_tai != kich_thuoc_du_kien:
        raise RuntimeError(f'Tai ffmpeg thieu: mong {kich_thuoc_du_kien} bytes, duoc {da_tai}.')


def cai_ffmpeg_on_dinh_moi_nhat() -> tuple[Path, Path]:
    may = platform.machine().lower()
    kien_truc = KIEN_TRUC_FFMPEG.get(may)
    if kien_truc is None:
        raise RuntimeError(f'Khong co ban ffmpeg cho kien truc {may}.')
    mau = re.compile(rf'^ffmpeg-n(?P<v>\d+\.\d+)-latest-{kien_truc}-gpl-(?P=v)\.tar\.xz$')

    release = goi_github_json(FFMPEG_RELEASE_API)
    ung_vien = []
    for tai_san in release.get('assets', []):
        m = mau.fullmatch(tai_san.get('name', ''))
        if m:
            ung_vien.append((tuple(int(p) for p in m.group('v').split('.')), tai_san))
    if not ung_vien:
        raise RuntimeError(f'Khong tim thay ban ffmpeg {kien_truc} on dinh nao.')
    _, tai_san = max(ung_vien, key=lambda x: x[0])
    url_tai = tai_san.get('browser_download_url', '')
    digest = tai_san.get('digest') or ''
    if not url_tai.startswith(FFMPEG_DOWNLOAD_PREFIX) or not digest.startswith('sha256:'):
        raise RuntimeError('URL/checksum ffmpeg khong hop le, tu choi tai ban khong xac minh duoc.')

    sha256_du_kien = digest.split(':', 1)[1].lower()
    kich_thuoc_du_kien = int(tai_san.get('size') or 0)
    FFMPEG_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    duong_dan_archive = FFMPEG_CACHE_DIR / tai_san['name']
    hop_le = (duong_dan_archive.is_file()
             and (not kich_thuoc_du_kien or duong_dan_archive.stat().st_size == kich_thuoc_du_kien)
             and sha256_file(duong_dan_archive) == sha256_du_kien)
    if hop_le:
        print(f'Dung ban ffmpeg da tai va xac minh: {duong_dan_archive.name}')
    else:
        print(f'Tai ban ffmpeg on dinh: {tai_san["name"]}')
        with tempfile.NamedTemporaryFile(dir=FFMPEG_CACHE_DIR, suffix='.download', delete=False) as tf:
            duong_dan_tam = Path(tf.name)
        try:
            tai_file(url_tai, duong_dan_tam, kich_thuoc_du_kien)
            if sha256_file(duong_dan_tam) != sha256_du_kien:
                raise RuntimeError('Checksum ffmpeg khong khop.')
            os.replace(duong_dan_tam, duong_dan_archive)
        finally:
            duong_dan_tam.unlink(missing_ok=True)

    FFMPEG_BIN_DIR.mkdir(parents=True, exist_ok=True)
    with tarfile.open(duong_dan_archive, mode='r:xz') as archive:
        thanh_vien = {}
        for m in archive.getmembers():
            p = Path(m.name)
            if m.isfile() and p.parent.name == 'bin' and p.name in {'ffmpeg', 'ffprobe'}:
                thanh_vien[p.name] = m
        if set(thanh_vien) != {'ffmpeg', 'ffprobe'}:
            raise RuntimeError('File nen ffmpeg thieu ffmpeg/ffprobe.')
        for ten, m in thanh_vien.items():
            nguon = archive.extractfile(m)
            dich = FFMPEG_BIN_DIR / ten
            dich_tam = FFMPEG_BIN_DIR / f'.{ten}.tmp'
            with nguon, dich_tam.open('wb') as out:
                shutil.copyfileobj(nguon, out)
            dich_tam.chmod(0o755)
            os.replace(dich_tam, dich)
    return FFMPEG_BIN_DIR / 'ffmpeg', FFMPEG_BIN_DIR / 'ffprobe'


ffmpeg_da_cai = FFMPEG_BIN_DIR / 'ffmpeg'
ffprobe_da_cai = FFMPEG_BIN_DIR / 'ffprobe'
ffmpeg_he_thong = shutil.which('ffmpeg')
ffprobe_he_thong = shutil.which('ffprobe')

if ffmpeg_da_cai.exists() or ffprobe_da_cai.exists():
    if ffmpeg_du_tinh_nang(ffmpeg_da_cai) and thuc_thi_duoc(ffprobe_da_cai):
        ffmpeg_bin, ffprobe_bin = ffmpeg_da_cai, ffprobe_da_cai
        print('Dung ban ffmpeg da cai rieng cho WanGP (du tinh nang).')
    else:
        print('Ban ffmpeg rieng cho WanGP thieu/loi -> tai lai...')
        ffmpeg_bin, ffprobe_bin = cai_ffmpeg_on_dinh_moi_nhat()
elif (ffmpeg_he_thong and ffprobe_he_thong
      and ffmpeg_du_tinh_nang(Path(ffmpeg_he_thong)) and thuc_thi_duoc(Path(ffprobe_he_thong))):
    ffmpeg_bin, ffprobe_bin = Path(ffmpeg_he_thong), Path(ffprobe_he_thong)
    print('Dung ffmpeg co san cua Colab (du tinh nang).')
else:
    print('ffmpeg cua Colab thieu hoac khong ho tro -fps_mode -> tai ban on dinh moi hon...')
    ffmpeg_bin, ffprobe_bin = cai_ffmpeg_on_dinh_moi_nhat()

if not ffmpeg_du_tinh_nang(ffmpeg_bin) or not thuc_thi_duoc(ffprobe_bin):
    raise RuntimeError('Xac minh ffmpeg that bai sau khi cai dat.')

os.environ['FFMPEG_BINARY'] = str(ffmpeg_bin.resolve())
os.environ['FFPROBE_BINARY'] = str(ffprobe_bin.resolve())
thu_muc_bin = str(ffmpeg_bin.resolve().parent)
cac_phan_path = [p for p in os.environ.get('PATH', '').split(os.pathsep) if p != thu_muc_bin]
os.environ['PATH'] = os.pathsep.join([thu_muc_bin, *cac_phan_path])
dong_phien_ban = subprocess.run([os.environ['FFMPEG_BINARY'], '-version'], check=True,
                                capture_output=True, text=True).stdout.splitlines()[0]
print(f'ffmpeg san sang: {dong_phien_ban}')


## Bước 5 — Cài PyTorch + các gói Python của WanGP

In [ ]:
# Buoc 5: cai PyTorch + toan bo requirements.txt cua WanGP bang `uv` (nhanh hon pip nhieu
# cho mot requirements.txt lon nhu the nay — WanGP la "super app" da nang video+anh+am
# thanh nen phu thuoc rat nhieu).
#
# Y CHINH (dung theo notebook cong dong Square-Zero-Labs/Wan2GP-on-Colab, DA GHI NHAN test
# tren Colab T4 free tier): TAI SU DUNG torch co san cua Colab neu no dung (>=2.7, KHONG
# phai 2.8.0/2.9.x — hai ban nay co loi ri bo nho da duoc cong dong WanGP ghi nhan), chi
# cai lai torch 2.10.0+cu128 khi ban co san khong dat. Tranh tai lai hang GB wheel CUDA da
# co san mot cach vo ich.
import json, os, subprocess, sys

PYTORCH_INDEX_URL = 'https://download.pytorch.org/whl/cu128'
TORCH_THAY_THE = ('torch==2.10.0', 'torchvision==0.25.0', 'torchaudio==2.10.0')
TORCH_KHONG_DUNG = ('2.8.0', '2.9.')
ONNXRUNTIME_GPU = 'onnxruntime-gpu==1.22.0'

_DO_TORCH = (
    'import json\n'
    'info = {}\n'
    'try:\n'
    '    import importlib.metadata as md, torch\n'
    '    for name in ("torch", "torchvision", "torchaudio"):\n'
    '        try: info[name] = md.version(name)\n'
    '        except Exception: pass\n'
    '    info["cuda"] = torch.cuda.is_available()\n'
    '    if info["cuda"]: info["device"] = torch.cuda.get_device_name(0)\n'
    'except Exception: pass\n'
    'print(json.dumps(info))\n'
)
_DO_ONNX = (
    'import json\n'
    'info = {}\n'
    'try:\n'
    '    import onnxruntime\n'
    '    info["version"] = onnxruntime.__version__\n'
    '    info["providers"] = onnxruntime.get_available_providers()\n'
    'except Exception: pass\n'
    'print(json.dumps(info))\n'
)

env = os.environ.copy()
if USE_GOOGLE_DRIVE_DATA:
    uv_cache = WAN_CACHE_DIR / 'uv'
    uv_cache.mkdir(parents=True, exist_ok=True)
    env['UV_CACHE_DIR'] = str(uv_cache)
    env['UV_LINK_MODE'] = 'copy'


def uv_pip(*args):
    subprocess.run([sys.executable, '-m', 'uv', 'pip', *args], check=True, env=env)


def uv_cai(*args):
    uv_pip('install', '--system', *args)


def do(ma):
    r = subprocess.run([sys.executable, '-c', ma], capture_output=True, text=True, env=env)
    try:
        return json.loads(r.stdout.strip().splitlines()[-1])
    except Exception:
        return {}


subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv'], check=True, env=env)

ban_torch = do(_DO_TORCH).get('torch', '')
try:
    dong_torch = tuple(int(p) for p in ban_torch.split('.')[:2])
except ValueError:
    dong_torch = ()
torch_dung_duoc = dong_torch >= (2, 7) and not ban_torch.startswith(TORCH_KHONG_DUNG)

if torch_dung_duoc:
    print(f'Dung PyTorch co san trong runtime ({ban_torch}).')
else:
    print('Cai PyTorch...')
    uv_cai(*TORCH_THAY_THE, '--index-url', PYTORCH_INDEX_URL)

print('Cai cac goi cua WanGP (co the mat vai phut, requirements.txt rat lon)...')
uv_cai('-r', str(WAN2GP_ROOT / 'requirements.txt'), '--index-strategy', 'unsafe-best-match')

# requirements.txt goc ghim mot ban onnxruntime-gpu NIGHTLY (tu mot feed rieng, de gay) —
# ghi de bang ban on dinh cong khai; khong co GPU provider thi lui ve CPU (van chay duoc,
# chi tach nen anh se cham hon).
uv_cai(ONNXRUNTIME_GPU)
onnx = do(_DO_ONNX)
if not onnx.get('version'):
    uv_pip('uninstall', '--system', 'onnxruntime-gpu')
    uv_cai('onnxruntime>=1.22.0')
    onnx = do(_DO_ONNX)

bao_cao = do(_DO_TORCH)
if not bao_cao.get('torch'):
    raise RuntimeError('PyTorch khong hoat dong sau khi cai. Vao Runtime > Restart session, '
                       'roi chay lai tu Buoc 1.')
if not bao_cao.get('cuda'):
    raise RuntimeError('Khong thay GPU. Vao Runtime > Change runtime type, chon GPU, '
                       'roi chay lai tu Buoc 1.')

print(f"San sang: PyTorch {bao_cao['torch']} tren {bao_cao['device']}.")
if 'CUDAExecutionProvider' not in onnx.get('providers', []):
    print('Luu y: tach nen anh (rembg) se chay tren CPU, khong tren GPU.')


## Bước 5b — Vá backend đồ hoạ cho Matanyone (Mask Generator)

In [ ]:
# Buoc 5b: vao Matanyone (cong cu Mask Generator ma ta se dung) dung backend do hoa
# 'TkAgg' — can GUI/man hinh that, KHONG chay duoc trong Colab headless -> se crash ngay
# khi mo Mask Generator. Vá sang 'Agg' (backend khong can man hinh).
from pathlib import Path

dich = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
tim = "matplotlib.use('TkAgg')"
thay = "matplotlib.use('Agg')"

if not dich.exists():
    print(f'Bo qua: khong thay {dich} (co the WanGP da doi cau truc thu muc).')
else:
    noi_dung = dich.read_text()
    if thay in noi_dung:
        print('Da la backend Agg tu truoc -> khong can sua.')
    elif tim in noi_dung:
        dich.write_text(noi_dung.replace(tim, thay, 1))
        print('Da thay TkAgg -> Agg trong interact_tools.py.')
    else:
        print('Khong tim thay dong goi backend -> khong sua gi (co the WanGP da doi code).')


## Bước 6 — Chạy WanGP (giữ cell này chạy, bấm vào link gradio.live xuất hiện trong log)

In [ ]:
# Buoc 6: chay giao dien Gradio. Doi dong "Running on public URL: https://xxxx.gradio.live"
# roi bam vao do de mo WanGP. GIU CELL NAY CHAY (dung bang nut Stop vuong) de link con song.
#
# --profile 5 = che do TOI THIEU HOA RAM HE THONG (khong phai VRAM) — T4 tren Colab
# thuong het RAM he thong (12,7 GB) truoc khi het VRAM (15 GB), nen day la lua chon dung
# theo tai lieu WanGP (docs/CLI.md) va da duoc cong dong kiem chung.
import os, subprocess, sys, threading, time

env = os.environ.copy()
env['WAN_CACHE_DIR'] = str(WAN_CACHE_DIR)
env['HF_HOME'] = str(WAN_CACHE_DIR / 'huggingface')
env['HUGGINGFACE_HUB_CACHE'] = str(WAN_CACHE_DIR / 'huggingface' / 'hub')
env['TORCH_HOME'] = str(WAN_CACHE_DIR / 'torch')
env['XDG_CACHE_HOME'] = str(WAN_CACHE_DIR / '.cache')

lenh = [sys.executable, '-u', 'wgp.py', '--listen', '--server-port', '7860', '--share', '--profile', '5']

print('Luu tru:', 'Google Drive' if USE_GOOGLE_DRIVE_DATA else 'o dia Colab (tam)')
print('Dang khoi dong WanGP...')
tien_trinh = subprocess.Popen(lenh, cwd=str(WAN2GP_ROOT), env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1)
dung_bao_hieu = threading.Event()


def _giu_song():
    while not dung_bao_hieu.is_set():
        time.sleep(45)
        if dung_bao_hieu.is_set():
            break
        print('[giu-song] cell van dang chay...')


_luong_giu_song = threading.Thread(target=_giu_song, daemon=True)
_luong_giu_song.start()

try:
    for dong in iter(tien_trinh.stdout.readline, ''):
        if not dong:
            break
        print(dong, end='')
except KeyboardInterrupt:
    print('Dang dung WanGP...')
    tien_trinh.terminate()
finally:
    dung_bao_hieu.set()
    tien_trinh.wait()
    _luong_giu_song.join(timeout=1)
    print(f'WanGP da dung (ma tra ve: {tien_trinh.returncode}).')


## Ghi chú vận hành / troubleshooting

- **Colab ngắt phiên do rảnh lâu**: chạy lại notebook từ Bước 1 — Bước 3 sẽ tự nhận ra
  repo/model đã tải (không tải lại từ đầu, đặc biệt nếu bạn bật `USE_GOOGLE_DRIVE_DATA`).
- **Hết VRAM/RAM khi Generate**: chọn model `Vace 1.3B` thay vì `14B`, giảm độ phân giải
  xuống 480p, giảm số khung hình. Xem cảnh báo VRAM ở cell đầu.
- **Muốn chất lượng cao hơn**: thử `Vace 14B` bản GGUF lượng tử hoá (WanGP tự tải đúng
  bản phù hợp phần cứng khi bạn chọn trong UI) — chấp nhận rủi ro chậm hơn/dễ hết VRAM
  hơn trên T4.
- **Mask Generator/Matanyone bị lỗi màn hình đen hoặc crash**: xác nhận Bước 5b đã chạy
  (vá `TkAgg` → `Agg`) — nếu vẫn lỗi, WanGP có thể đã đổi cấu trúc thư mục, tìm file
  `interact_tools.py` mới trong repo và áp lại thủ công.
- **Muốn dùng lại pipeline tự viết (ComfyUI + VACE) thay vì WanGP**: xem
  `VideoObjectSwap_WanVACE_Colab.ipynb` — vẫn giữ lại trong thư mục làm tài liệu tham
  khảo cơ chế VACE, nhưng khuyến nghị dùng notebook WanGP này vì đã được kiểm chứng nhiều
  hơn.
- **Muốn chạy hàng loạt không qua UI**: WanGP hỗ trợ `python wgp.py --process
  my_queue.zip` (tạo queue trong UI, lưu lại, rồi xử lý headless) — xem
  [docs/CLI.md](https://github.com/deepbeepmeep/Wan2GP/blob/main/docs/CLI.md) của WanGP.

## Chưa làm ở bản này

1. Chưa đo thời gian cài đặt/sinh video thật trên T4 — cấu hình dựa trên notebook cộng
   đồng đã test (Square-Zero-Labs/Wan2GP-on-Colab) + tài liệu chính thức của WanGP, chưa
   phải số đo của chính phiên bản notebook này.
2. Chưa tự động hoá bước "click chọn vùng sản phẩm" — đây là thao tác tay trong UI
   (Mask Generator), đúng theo thiết kế của WanGP (thay cho việc tự đoán toạ độ bbox như
   notebook cũ).
